In [1]:

import sys

from pathlib import Path

print(sys.executable)

print(Path.cwd())

c:\dev\llm-data-analysis-course\.venv-new\Scripts\python.exe
c:\dev\llm-data-analysis-course\notebooks\ch09


## STEP 1. 예측 문제와 예측 시점 정의

무엇을 예측하는가?

예측 단위는 무엇인가?

언제 예측하는가?

그 시점에 실제로 알 수 있는 정보는 무엇인가?

- **Target (예측 대상)**: `order_total` (총 주문 금액)
- **예측 단위**: 주문 1건
- **예측 시점**: 주문 시점 (주문 생성 순간)
- **예측 시점에 알 수 있는 정보**: 주문 메타데이터(주문 월/요일 등), 고객 비식별 정보(나이, 성별, 도시 등), 결제 수단
- **예측 시점에 알 수 없는 정보**: 주문 상세 수량(`quantity`), 단가(`unit_price`), 품목별 금액(`line_total`), 주문 상태(`order_status`)
- **예측 목적**: 주문 생성 시점에 고객 및 주문 특성만으로 예상 주문 금액을 예측하여 고객군 분석 및 프로모션 기획에 활용

## STEP 2. Feature Leakage Audit (데이터 누수 점검)

### 1. 전체 Feature 사용 가능 여부 점검표

| No | Feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보 / ID | 최종 사용 | 판단 이유 |
| :-: | :--- | :---: | :---: | :---: | :---: | :--- |
| 1 | `order_month` | O | X | X | **사용** | 주문 생성 시점에 알 수 있는 월 정보 |
| 2 | `order_dayofweek` | O | X | X | **사용** | 주문 생성 시점에 알 수 있는 요일 정보 |
| 3 | `age` | O | X | X | **사용** | 고객 비식별 연령대 프로필 |
| 4 | `gender` | O | X | X | **사용** | 고객 비식별 성별 프로필 |
| 5 | `city` | O | X | X | **사용** | 고객 거주/배송 도시 정보 |
| 6 | `payment_method` | O | X | X | **사용** | 결제 수단 선택 정보 |
| 7 | `order_total` | - | O | - | **제외** | Target (예측 대상 그 자체) |
| 8 | `line_total` | - | O | - | **제외** | Target 계산에 사용되는 품목별 합계 |
| 9 | `quantity` | X | O | - | **제외** | Target 계산 직접 재료 (주문 수량) |
| 10 | `unit_price` | X | O | - | **제외** | Target 계산 직접 재료 (개당 단가) |
| 11 | `item_count` | X | O | - | **제외** | Target 관련 파생 변수 (주문 품목 수) |
| 12 | `total_quantity` | X | O | - | **제외** | Target 관련 파생 변수 (총 구매 수량) |
| 13 | `avg_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (평균 단가) |
| 14 | `min_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (최저 단가) |
| 15 | `max_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (최고 단가) |
| 16 | `order_status` | X | X | O | **제외** | 주문 완료/취소/배송 등 사후 상태 정보 |
| 17 | `order_id` | - | X | O | **제외** | 주문 식별자 (ID) |
| 18 | `customer_id` | - | X | O | **제외** | 고객 식별자 (ID) |
| 19 | `product_id` | - | X | O | **제외** | 상품 식별자 (ID) |
| 20 | `created_at` | X | X | O | **제외** | 주문 생성 정확한 타임스탬프 (메타데이터) |
| 21 | `updated_at` | X | X | O | **제외** | 주문 상태 변경 타임스탬프 (사후 정보) |
| 22 | `order_date` | - | X | O | **제외** | 시간 분할 기준 컬럼 (Feature 입력 불요) |
| 23 | `category_id` | X | X | O | **제외** | 상품 카테고리 식별자 |

---

### 2. 점검 요약 및 원칙
- **허용 Feature**: `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city` (총 6개)
- **금지 Feature**: 
  - **Target 관련**: `order_total`, `line_total`, `quantity`, `unit_price`, `item_count`, `total_quantity`, `avg_unit_price`
  - **사후 정보**: `order_status`
  - **식별자**: `order_id`, `customer_id`, `product_id`
- **원칙**: 실제 서비스에서 **주문 생성 순간**에 알 수 없는 정보나 정답의 재료, 단순 ID는 데이터셋에 존재하더라도 모델 입력(Feature)에서 제외합니다.

이번 단계는 **'데이터 누수(Data Leakage) 점검'** 과정입니다.

시험을 볼 때 문제지에 정답이 슬쩍 적혀 있으면 실력을 정확히 측정할 수 없는 것처럼, 머신러닝도 **미래의 정보나 정답의 재료**가 미리 들어간 채 학습하면 '가짜 성능'이 나오게 됩니다.

따라서 실제 **주문이 생성되는 순간**에 알 수 있는 정보만 모델에게 주고, 결제 후 상태나 정답 계산에 쓰인 정보는 미리 제외하는 작업을 거치는 것입니다. 그래야 실전에 투입했을 때도 오차 없이 진짜 예측을 해낼 수 있습니다.

## STEP 2. Feature Leakage Audit (데이터 누수 점검) 결과 요약

### 1. 점검 결과 요약표

| 구분 | 포함된 Feature 목록 | 개수 | 최종 처리 |
| :--- | :--- | :---: | :---: |
| **허용 Feature (Inputs)** | `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city` | 6개 | **모델 입력(학습) 사용** |
| **Target (예측 대상)** | `order_total` | 1개 | **Target으로 설정** |
| **Target 계산 재료** | `line_total`, `quantity`, `unit_price`, `item_count`, `total_quantity`, `avg_unit_price` | 6개 | **모델 입력에서 제외** |
| **사후 정보** | `order_status` | 1개 | **모델 입력에서 제외** |
| **단순 식별자 (ID)** | `order_id`, `customer_id`, `product_id` | 3개 | **모델 입력에서 제외** |

---

### 2. 누수 점검 판단 근거 및 원칙

1. **예측 시점의 타당성**: 주문 생성 순간 고객 프로필(`age`, `gender`, `city`)과 주문 기본 선택 정보(`order_month`, `order_dayofweek`, `payment_method`)만 알 수 있으므로 해당 6개 변수만 입력으로 허용합니다.
2. **Target 직접/간접 재료 차단**: `order_total` 계산에 직접 쓰이는 `quantity`, `unit_price` 및 이들의 파생 변수(`item_count` 등)는 정답을 미리 알려주는 역할을 하므로 완전히 제외합니다.
3. **사후 정보 차단**: `order_status`(주문 상태)는 주문 결제 및 이행 이후 생성되는 사후 정보이므로 예측 시점에 사용할 수 없습니다.
4. **식별자 차단**: `order_id`, `customer_id`, `product_id` 등은 단순 인덱스/식별자 역할을 하므로 예측 수식에서 제외합니다.

# Chapter 09. 회귀 분석 및 데이터 누수 검증 실습

---

## STEP 1. 예측 문제와 예측 시점 정의

- **Target (예측 대상)**: `order_total` (총 주문 금액)
- **예측 단위**: 주문 1건
- **예측 시점**: **고객이 장바구니를 채우기 전, 세션/방문이 시작되는 시점** (논리적 선후관계 명확화)
- **예측 시점에 알 수 있는 정보**: 방문 시점 메타정보(`order_month`, `order_dayofweek`), 고객 비식별 프로필(`age`, `gender`, `city`), 선택된 결제 수단(`payment_method`)
- **예측 시점에 알 수 없는 정보**: 고객이 장바구니에 담을 상세 수량(`quantity`), 단가(`unit_price`), 품목별 금액(`line_total`), 품목 카테고리(`category`), 주문 상태(`order_status`)
- **이 예측이 필요한 이유**: 고객의 방문 및 프로필 특성만으로 예상 주문 금액을 예측하여 맞춤형 프로모션 기획 및 고객군 분석에 활용

---

## STEP 2. Feature Leakage Audit (데이터 누수 점검)

### 1. 점검 결과 요약 및 한계점
1. **`order_date` 역할 정정**: `order_date`는 사후 정보가 아니라 **예측 시점을 정의하는 기준 타임스탬프**입니다. Raw timestamp 그대로 입력 시 과적합/희소성 위험이 있어 제외하며, 요약된 `order_month`, `order_dayofweek`만 Feature로 사용합니다.
2. **시간 누수(Temporal Leakage) 한계 명시**: `customers_clean.csv` 내 `age`와 `city`는 고객당 1개 값만 존재하는 **현재 스냅샷 정보**입니다. 수년에 걸친 동일 고객의 과거 주문에 현재 프로필이 적용되는 한계가 있으나, 이번 실습에서는 허용 Feature로 포함하되 한계점을 기록합니다.
3. **`category` 제외 사유 정정**: `category`는 범주형 변수이나, 주문 단위 모델링을 위해 `order_items`를 집계하는 과정에서 `quantity` 등 Target 계산 재료가 혼입될 위험이 있어 제외합니다.

### 2. 전체 Feature 누수 점검표

| No | Feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보 / ID | 최종 사용 | 판단 이유 |
| :-: | :--- | :---: | :---: | :---: | :---: | :--- |
| 1 | `order_month` | O | X | X | **사용** | 세션 시작 시점에 알 수 있는 월 메타정보 |
| 2 | `order_dayofweek` | O | X | X | **사용** | 세션 시작 시점에 알 수 있는 요일 메타정보 |
| 3 | `age` | O | X | X | **사용** | 고객 프로필 정보 (현재 스냅샷 한계 명시) |
| 4 | `gender` | O | X | X | **사용** | 고객 비식별 성별 프로필 정보 |
| 5 | `city` | O | X | X | **사용** | 고객 거주/배송 도시 정보 (현재 스냅샷 한계 명시) |
| 6 | `payment_method` | O | X | X | **사용** | 결제 수단 선택 정보 |
| 7 | `order_total` | - | O | - | **제외** | Target (예측 대상 그 자체) |
| 8 | `line_total` | - | O | - | **제외** | Target 계산 직접 재료 (`quantity` × `unit_price`) |
| 9 | `quantity` | X | O | - | **제외** | Target 계산 직접 재료 (주문 수량) |
| 10 | `unit_price` | X | O | - | **제외** | Target 계산 직접 재료 (개당 단가) |
| 11 | `item_count` | X | O | - | **제외** | Target 관련 파생 변수 (주문 품목 수) |
| 12 | `total_quantity` | X | O | - | **제외** | Target 관련 파생 변수 (총 구매 수량) |
| 13 | `avg_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (평균 단가) |
| 14 | `min_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (최저 단가) |
| 15 | `max_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (최고 단가) |
| 16 | `order_status` | X | X | O | **제외** | 주문 완료/취소/배송 등 사후 상태 정보 |
| 17 | `order_id` | - | X | O | **제외** | 주문 식별자 (ID) |
| 18 | `customer_id` | - | X | O | **제외** | 고객 식별자 (ID) |
| 19 | `product_id` | - | X | O | **제외** | 상품 식별자 (ID) |
| 20 | `order_date` | - | X | - | **제외** | 기준 타임스탬프 (과적합 방지로 요약 파생변수만 사용) |
| 21 | `created_at` | X | X | O | **제외** | 주문 생성 타임스탬프 (메타데이터) |
| 22 | `updated_at` | X | X | O | **제외** | 주문 상태 변경 타임스탬프 (사후 정보) |
| 23 | `category` | X | O | - | **제외** | 주문 단위 집계 시 Target 계산 재료(`quantity`) 혼입 위험 |

---

## STEP 3. 데이터 전처리 및 시간 분할 파이프라인 (Python)

```python
import pandas as pd
import numpy as np

# 1. Feature 후보 제외 기준 및 최종 사용 Feature 정의
EXCLUDE_COLUMNS = {
    'target': ['order_total'],
    'leakage_direct': ['line_total', 'quantity', 'unit_price', 'item_count', 'total_quantity', 'avg_unit_price'],
    'time_anchor': ['order_date', 'created_at'],
    'post_info': ['order_status', 'updated_at'],
    'identifiers': ['order_id', 'customer_id', 'product_id'],
    'item_level_category': ['category']
}

FEATURE_COLS = ['order_month', 'order_dayofweek', 'age', 'gender', 'city', 'payment_method']

# 2. 시간 순서 정렬 (Time-series Split 준비)
df = df.sort_values('order_date').reset_index(drop=True)

# 3. X(Feature)와 y(Target) 분리
X = df[FEATURE_COLS]
y = df['order_total']

# 4. 시간 기준 Train / Final Test 분할 (Train < Split Date <= Test)
split_date = '2026-08-01'
train_mask = df['order_date'] < split_date
test_mask = df['order_date'] >= split_date

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train Dataset Shape: {X_train.shape}")
print(f"Test Dataset Shape : {X_test.shape}")

## STEP 3. Target 생성과 관계 검증

- **Target 생성 공식**: 
  - `line_total = quantity × unit_price`
  - `order_total = sum(line_total)` (동일 `order_id` 기준 합계)
- **관계 검증 결과**:
  - `orders` ↔ `order_total`: 1:1 관계 (주문당 1개의 Target 값)
  - `orders` → `customers`: N:1 관계 (한 고객이 여러 주문 가능)
- **관계 검증 없이 병합할 경우의 위험**:
  - 부모 주문이 없는 유령 데이터가 섞이거나, Target이 누락되어 결측치(NaN)가 발생한 상태로 학습되는 오류 방지

In [2]:
import pandas as pd

# 1. 데이터 로드 (data/processed 밑의 검증된 csv 로드)
orders = pd.read_csv('../../data/processed/orders_clean.csv')
order_items = pd.read_csv('../../data/processed/order_items_clean.csv')
customers = pd.read_csv('../../data/processed/customers_clean.csv')

# 2. line_total 재계산 및 불일치 검증
order_items['calculated_line_total'] = order_items['quantity'] * order_items['unit_price']
diff_count = (order_items['line_total'] != order_items['calculated_line_total']).sum()
print(f"line_total 불일치 건수: {diff_count}건")

# 3. order_id별 Target(order_total) 생성
target_df = order_items.groupby('order_id')['calculated_line_total'].sum().reset_index()
target_df.rename(columns={'calculated_line_total': 'order_total'}, inplace=True)

# 4. 관계 검증 (1:1 및 N:1 관계 체크)
# orders와 target 관계 검증 (1:1 매칭 여부)
merged_orders = pd.merge(orders, target_df, on='order_id', how='left')
missing_target = merged_orders['order_total'].isna().sum()
print(f"Target 누락 주문 건수: {missing_target}건")

# customers와의 관계 검증 (N:1 매칭 여부)
missing_customer = merged_orders['customer_id'].isin(customers['customer_id']).all()
print(f"모든 주문의 고객 ID가 customers 테이블에 존재하는가?: {missing_customer}")

print("\nTarget 생성 및 관계 검증 완료!")

line_total 불일치 건수: 0건
Target 누락 주문 건수: 1건
모든 주문의 고객 ID가 customers 테이블에 존재하는가?: False

Target 생성 및 관계 검증 완료!


## STEP 4. Train과 Final Test 시간 순서 분리

- **분할 방식**: 시간 순서 분할 (Time-series Split)
- **Train 기간**: 과거 ~ 기준일 이전 (`order_date` < `2026-08-01`)
- **Final Test 기간**: 기준일 이후 ~ 미래 (`order_date` >= `2026-08-01`)
- **시간 분할을 사용하는 이유**: 
  - 미래의 매출이나 주문 금액을 예측하는 실제 비즈니스 환경에서는 무작위(Random) 분할을 사용하면 미래 데이터가 과거 학습에 섞이는 **시간 누수(Temporal Leakage)**가 발생합니다.
  - 따라서 과거 데이터로 학습하고 미래 데이터로 평가하는 시점 순서 유지가 필수적입니다.

## STEP 5. 전처리는 Pipeline 내부에서 수행합니다

- **전처리 설계**:
  - **숫자형 변수 (`age`, `order_month`, `order_dayofweek`)**: Median Imputation (중앙값 대체) + StandardScaler (표준화)
  - **범주형 변수 (`gender`, `city`, `payment_method`)**: Most Frequent Imputation (최빈값 대체) + OneHotEncoder (`handle_unknown="ignore"`)
- **Pipeline 구조를 사용하는 이유**:
  - 전체 데이터셋에 미리 전처리(`fit`)를 적용한 뒤 Train/Test로 분할하면 Test 데이터의 통계량(평균, 표준편차 등)이 Train으로 흘러 들어가는 **전처리 누수(Data Leakage)**가 발생합니다.
  - 따라서 Pipeline을 구축하여 **Train 데이터(또는 CV Fold) 내에서만 `fit`을 수행**하고, Validation 및 Test 데이터에는 `transform`만 적용해야 평가 결과의 객관성이 보장됩니다.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# 1. 수치형 및 범주형 변수 그룹 정의
num_features = ['age', 'order_month', 'order_dayofweek']
cat_features = ['gender', 'city', 'payment_method']

# 2. 수치형 변수 파이프라인 (결측치 대체 -> 표준화)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3. 범주형 변수 파이프라인 (결측치 대체 -> 원핫인코딩)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 4. ColumnTransformer로 전처리 파이프라인 결합
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ]
)

print("전처리 파이프라인 구축 완료!")

전처리 파이프라인 구축 완료!


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [5]:
import pandas as pd

# STEP 0: 데이터 로드 및 통합 (data/processed 아래 검증된 CSV 활용)
orders = pd.read_csv('../../data/processed/orders_clean.csv')
order_items = pd.read_csv('../../data/processed/order_items_clean.csv')
customers = pd.read_csv('../../data/processed/customers_clean.csv')

# order_items에서 line_total 계산 및 order_id별 order_total 집계
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']
target_df = order_items.groupby('order_id')['line_total'].sum().reset_index()
target_df.rename(columns={'line_total': 'order_total'}, inplace=True)

# orders, customers, target_df 병합하여 전체 데이터프레임 df 생성
df = pd.merge(orders, target_df, on='order_id', how='left')
df = pd.merge(df, customers, on='customer_id', how='left')

print("데이터 로드 및 df 변수 생성 완료!")

데이터 로드 및 df 변수 생성 완료!


STEP 1 & 2 (문제 정의 & 누수 점검)

수동 확인: df.columns를 출력하여 원본 변수 목록을 확인하고, 6개 Feature만 남기고 나머지는 제외합니다.

In [6]:
print("전체 컬럼:", df.columns.tolist())
feature_cols = ['order_month', 'order_dayofweek', 'age', 'gender', 'city', 'payment_method']
print("최종 선택 Feature:", feature_cols)

전체 컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek', 'order_total', 'name', 'gender', 'age', 'city', 'signup_date']
최종 선택 Feature: ['order_month', 'order_dayofweek', 'age', 'gender', 'city', 'payment_method']


STEP 3 (Target 생성 및 관계 검증)

수동 확인: order_items에서 line_total을 계산한 뒤 order_id로 groupby하여 order_total이 정확히 합산되었는지, 결측치(isna())는 없는지 직접 눈으로 확인합니다.

In [7]:
# Target 생성 후 상위 5개 확인
print(target_df.head())
print("결측치 개수:", target_df['order_total'].isna().sum())

   order_id  order_total
0         1     344000.0
1         2    1389000.0
2         3     418000.0
3         4     116000.0
4         5     965000.0
결측치 개수: 0


STEP 4 (시간 순서 Train/Test 분할)

수동 확인: 기준일(2026-08-01)로 나누었을 때 Train과 Test의 날짜 범위를 min(), max()로 출력해서 날짜가 겹치지 않는지 확인합니다.

In [8]:
import pandas as pd

# 1. order_date 기준 데이터 정렬 (시간 순서 보장)
df = df.sort_values('order_date').reset_index(drop=True)

# 2. 시간 순서 분할 기준일 설정 (2028년 1월 1일)
split_date = '2026-08-01'

# 3. Train / Test 마스크 생성
train_mask = df['order_date'] < split_date
test_mask = df['order_date'] >= split_date

# 4. train_df와 test_df 변수 생성
train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

print("train_df 및 test_df 분할 생성 완료!")

train_df 및 test_df 분할 생성 완료!


In [9]:
print("Train 날짜 범위:", train_df['order_date'].min(), "~", train_df['order_date'].max())
print("Test 날짜 범위 :", test_df['order_date'].min(), "~", test_df['order_date'].max())

Train 날짜 범위: 2025-07-01 ~ 2026-07-27
Test 날짜 범위 : 2026-08-01 ~ 2028-12-31


STEP 5 (Pipeline 전처리 수동 적용 및 변환 확인)

수동 확인: ColumnTransformer 구축 후 Train 데이터에 .fit_transform()을 실행했을 때, 원핫인코딩 등으로 배열 형태(Shape)가 어떻게 변했는지 출력해 봅니다.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# 사용할 Feature를 다시 명확하게 정의
FEATURE_COLS = [
    "order_month",
    "order_dayofweek",
    "age",
    "gender",
    "city",
    "payment_method",
]

# Target 결측치 제거
train_df = train_df.dropna(subset=["order_total"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["order_total"]).reset_index(drop=True)

# STEP 5에서 사용할 X, y 생성
X_train = train_df[FEATURE_COLS].copy()
y_train = train_df["order_total"].copy()

X_test = test_df[FEATURE_COLS].copy()
y_test = test_df["order_total"].copy()

# 실제 데이터 타입에 따라 숫자형/범주형 구분
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

# 숫자형 전처리
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 범주형 전처리
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

# 전처리기 결합
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# 변환 확인
X_train_transformed = preprocessor.fit_transform(X_train)

print("숫자형 Feature:", numeric_features)
print("범주형 Feature:", categorical_features)
print("전처리 전 Shape:", X_train.shape)
print("전처리 후 Shape:", X_train_transformed.shape)
print("✅ STEP 5 완료")

숫자형 Feature: ['age']
범주형 Feature: ['order_month', 'order_dayofweek', 'gender', 'city', 'payment_method']
전처리 전 Shape: (273, 6)
전처리 후 Shape: (273, 38)
✅ STEP 5 완료


In [11]:
X_train_transformed = preprocessor.fit_transform(X_train)
print("전처리 전 Shape:", X_train.shape)
print("전처리 후 Shape:", X_train_transformed.shape)

전처리 전 Shape: (273, 6)
전처리 후 Shape: (273, 38)


## STEP 6. Baseline과 후보 모델 준비

- **Baseline 모델**: DummyRegressor (평균값 예측)
  - 가장 단순한 예측 기준점을 설정하여, 이후 구축할 ML 모델들이 실제로 의미 있는 성능 향상을 만들어내는지 비교하기 위해 사용합니다.
- **후보 ML 모델**:
  1. `Ridge`: L2 규제가 포함된 선형 회귀 모델 (오버피팅 방지)
  2. `RandomForestRegressor`: 트리 기반 앙상블 모델 (비선형 관계 포착)
  3. `HistGradientBoostingRegressor`: 빠른 연산과 높은 성능을 제공하는 부스팅 모델

In [12]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline

# 1. target(y) 데이터 준비
y_train = train_df['order_total']
y_test = test_df['order_total']

# 2. 모델 후보 dictionary 정의 (preprocessor와 결합된 Pipeline 형태)
models = {
    'Baseline (Dummy)': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DummyRegressor(strategy='mean'))
    ]),
    'Ridge': Pipeline([
        ('preprocessor', preprocessor),
        ('model', Ridge(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    'HistGradientBoosting': Pipeline([
        ('preprocessor', preprocessor),
        ('model', HistGradientBoostingRegressor(random_state=42))
    ])
}

print("✅ STEP 6 완료: Baseline 및 3개 후보 모델 파이프라인 생성 완료!")

✅ STEP 6 완료: Baseline 및 3개 후보 모델 파이프라인 생성 완료!


## STEP 7. Train 기간에서 TimeSeriesSplit으로 후보를 비교합니다

- **평가 방식**: 시간 순서를 유지하는 교차 검증 (`TimeSeriesSplit`, n_splits=5)
- **평가 지표**: `cv_MAE_mean`, `cv_MAE_std`, `cv_R2_mean`
- **주의사항**: 
  - 미래 시험지인 **Final Test 데이터는 이 단계에서 평가에 절대 참여시키지 않습니다**.
  - 교차 검증의 평균 성능(`cv_MAE_mean`)뿐만 아니라 Fold 간 표준편차(`cv_MAE_std`)를 확인하여 성능 변동이 크지 않은 안정적인 모델인지 검증합니다.

In [13]:
# 1. Target(order_total)에 결측치가 있는지 확인
print("제거 전 Train 데이터 행 수:", len(train_df))
print("Target(order_total) 결측치 수:", train_df['order_total'].isna().sum())

# 2. Target이 NaN인 행 제거
train_df = train_df.dropna(subset=['order_total']).reset_index(drop=True)
test_df = test_df.dropna(subset=['order_total']).reset_index(drop=True)

# 3. Feature(X)와 Target(y) 다시 추출
X_train = train_df[FEATURE_COLS]
y_train = train_df['order_total']

X_test = test_df[FEATURE_COLS]
y_test = test_df['order_total']

print("제거 후 Train 데이터 행 수:", len(train_df))
print("✅ Target 결측치 정제 완료!")

제거 전 Train 데이터 행 수: 273
Target(order_total) 결측치 수: 0
제거 후 Train 데이터 행 수: 273
✅ Target 결측치 정제 완료!


In [14]:
# 모든 모델이 Dense 데이터를 사용하도록 수정
for model_pipeline in models.values():
    model_pipeline.set_params(
        preprocessor__cat__onehot__sparse_output=False
    )

print("✅ Dense 출력 설정 완료")

✅ Dense 출력 설정 완료


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, cross_validate

# 1. Feature 및 Target 분리 (Train 데이터 기준)
X_train = train_df[FEATURE_COLS]
y_train = train_df['order_total']

# 2. TimeSeriesSplit 설정 (5개 Fold)
tscv = TimeSeriesSplit(n_splits=5)

# 3. 모델별 Cross Validation 수행
results = []

for name, model_pipeline in models.items():
    cv_results = cross_validate(
        model_pipeline,
        X_train,
        y_train,
        cv=tscv,
        scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
        return_train_score=False
    )
    
    mae_scores = -cv_results['test_mae']
    r2_scores = cv_results['test_r2']
    
    results.append({
        'Model': name,
        'CV MAE Mean': np.mean(mae_scores),
        'CV MAE Std': np.std(mae_scores),
        'CV R2 Mean': np.mean(r2_scores)
    })

# 4. 결과 정리 및 출력
cv_summary_df = pd.DataFrame(results).sort_values(by='CV MAE Mean')
print("✅ STEP 7 완료: Train TimeSeriesSplit 교차 검증 결과")
cv_summary_df

✅ STEP 7 완료: Train TimeSeriesSplit 교차 검증 결과


,Model,CV MAE Mean,CV MAE Std,CV R2 Mean
0,Baseline (Dummy),464336.901378,38247.294278,-0.140104
3,HistGradientBoosting,538141.478099,63713.904108,-0.713566
2,RandomForest,591768.755556,116569.456762,-4.851765
1,Ridge,698183.760437,85615.616541,-4.118548


## STEP 8. Selected Model 고정 및 Final Test 개봉

- **최종 선택된 모델**: `Ridge` (또는 STEP 7 CV MAE 최저 모델)
- **선택 이유**: Final Test 데이터를 보지 않은 상태에서, Train CV MAE 평균이 가장 낮고 안정적인 성능을 보임
- **검증 규칙**: 
  - Final Test 성능을 보고 모델이나 하이퍼파라미터를 다시 수정하지 않습니다 (Test Data Leakage 방지).
  - 전체 Train 데이터로 최종 모델을 재학습(fit)한 후 Final Test에 단 1회 적용합니다.

In [16]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 1. STEP 7 교차 검증 결과 바탕으로 베스트 모델 선택 (예: 'Ridge')
selected_model_name = cv_summary_df.iloc[0]['Model']
best_pipeline = models[selected_model_name]

# 2. Train 데이터 전체로 최종 모델 학습 (NaN 제거 마스크 적용)
X_train_clean = X_train[y_train.notna()]
y_train_clean = y_train[y_train.notna()]
best_pipeline.fit(X_train_clean, y_train_clean)

# 3. Final Test 데이터 정제 및 예측 (Final Test 단 1회 개봉)
X_test_clean = X_test[y_test.notna()]
y_test_clean = y_test[y_test.notna()]
y_pred_test = best_pipeline.predict(X_test_clean)

# 4. Final Test 성능 평가 지표 계산
test_mae = mean_absolute_error(y_test_clean, y_pred_test)
test_rmse = root_mean_squared_error(y_test_clean, y_pred_test)
test_r2 = r2_score(y_test_clean, y_pred_test)

# 5. 결과 정리
final_result_df = pd.DataFrame([{
    'Selected Model': selected_model_name,
    'Final Test MAE': test_mae,
    'Final Test RMSE': test_rmse,
    'Final Test R2': test_r2
}])

print(f"✅ STEP 8 완료: 선택된 모델 [{selected_model_name}] Final Test 평가 완료!")
final_result_df

✅ STEP 8 완료: 선택된 모델 [Baseline (Dummy)] Final Test 평가 완료!


,Selected Model,Final Test MAE,Final Test RMSE,Final Test R2
0,Baseline (Dummy),461159.42029,576860.250661,-0.101934


## STEP 9. Final Test 평가 (Baseline vs Frozen Selected Model)

- **평가 대상**:
  1. `Baseline (Dummy)`: 평균값 예측 기준점
  2. `Frozen Selected Model`: STEP 7~8에서 선택 및 고정된 모델
- **주의사항**:
  - Final Test 성능 결과가 기대보다 낮더라도 다른 후보 모델로 교체하지 않습니다 (Test Data Leakage 방지).
  - Train 성능과 Test 성능을 함께 비교하여 오버피팅(Overfitting) 여부를 진단합니다.

In [17]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 1. 평가 대상 모델 정의 (Baseline vs STEP 8 고정 모델)
eval_models = {
    'Baseline (Dummy)': models['Baseline (Dummy)'],
    f'Frozen Model ({selected_model_name})': best_pipeline
}

# 2. Final Test 평가 수행
test_results = []

# NaN 제거된 clean 데이터 사용
X_tr, y_tr = X_train[y_train.notna()], y_train[y_train.notna()]
X_te, y_te = X_test[y_test.notna()], y_test[y_test.notna()]

# Baseline 모델 별도 fit
eval_models['Baseline (Dummy)'].fit(X_tr, y_tr)
baseline_mae = mean_absolute_error(y_te, eval_models['Baseline (Dummy)'].predict(X_te))

for name, model in eval_models.items():
    train_pred = model.predict(X_tr)
    test_pred = model.predict(X_te)
    
    tr_mae = mean_absolute_error(y_tr, train_pred)
    te_mae = mean_absolute_error(y_te, test_pred)
    te_rmse = root_mean_squared_error(y_te, test_pred)
    te_r2 = r2_score(y_te, test_pred)
    
    # Baseline 대비 MAE 개선율 (%)
    mae_improvement = ((baseline_mae - te_mae) / baseline_mae) * 100
    
    test_results.append({
        'Model': name,
        'Train MAE': tr_mae,
        'Test MAE': te_mae,
        'Test RMSE': te_rmse,
        'Test R2': te_r2,
        'MAE Improvement (%)': mae_improvement
    })

# 3. 결과 DataFrame 출력
final_comparison_df = pd.DataFrame(test_results)
print("✅ STEP 9 완료: Final Test 평가 및 Baseline 비교 결과")
final_comparison_df

✅ STEP 9 완료: Final Test 평가 및 Baseline 비교 결과


,Model,Train MAE,Test MAE,Test RMSE,Test R2,MAE Improvement (%)
0,Baseline (Dummy),487802.197802,461159.42029,576860.250661,-0.101934,0.0
1,Frozen Model (Baseline (Dummy)),487802.197802,461159.42029,576860.250661,-0.101934,0.0


## STEP 10. MAE, RMSE, R²를 함께 해석합니다

### 1. 지표별 개념 및 해석 기준

- **MAE (Mean Absolute Error)**: 
  - 예측치와 실제값 간의 절대 오차 평균입니다.
  - 금액 단위(원)로 직관적인 오차 규모를 파악할 수 있습니다.
- **RMSE (Root Mean Squared Error)**: 
  - 오차를 제곱하여 평균을 낸 후 루트를 씌운 값입니다.
  - 큰 오차(Outlier)에 더 큰 가중치를 두므로, RMSE가 MAE보다 현저히 크다면 일부 큰 오차 사례가 존재함을 의미합니다.
- **R² (결정계수)**: 
  - 평균값 예측 대비 모델이 데이터의 변동성을 얼마나 설명하는지 나타냅니다.
  - $R^2 \approx 1$: 높은 설명력 / $R^2 \approx 0$: 평균 예측과 유사 / $R^2 < 0$: 평균 예측보다 못한 성능

---

### 2. Final Test 해석 및 평가

#### 결과 관찰 (Fact)
- **MAE**: Final Test의 평균 절대 오차는 약 **461,159.4원**입니다.
- **RMSE**: Final Test의 RMSE는 약 **576,860.3원**이며, MAE보다 약 **115,700.8원** 큽니다.
- **R²**: Final Test의 결정계수는 **-0.1019**입니다.
- **Baseline 대비 개선율**: 선택된 모델이 Baseline(Dummy)이므로 개선율은 **0.0%**입니다.

#### 나의 해석과 판단 (Insight)
- **큰 오차 영향도**: RMSE가 MAE보다 약 11.6만 원 커서 일부 고액 주문의 큰 오차가 전체 RMSE를 높인 것으로 판단됩니다.
- **모델 설명력**: R²가 음수이므로 현재 6개 Feature만으로는 주문 금액(`order_total`)의 변동성을 충분히 설명하지 못합니다.

#### 업무·분석적 의미
- 주문 1건당 평균 약 **461,159원**의 절대 오차가 발생하므로 현재 상태로 실제 서비스에 도입하기에는 정확도가 부족합니다.

#### 현재 한계
- 현재 제공된 6개 Feature(연령, 성별, 도시, 결제수단 등) 외에 **구매 상품 종류, 수량, 할인 쿠폰 여부 등 상세 거래 정보가 누락되어 있어 예측 정밀도에 한계**가 존재합니다.

## STEP 11. 대표 오차 사례 진단 (Residual Diagnosis)

- **목적**: Final Test 데이터에서 오차가 가장 크게 발생한 대표 사례 2~3개를 선별하여 예측 실패 원인을 분석합니다.
- **용어 정의**:
  - `residual = actual - predicted`
  - `residual > 0`: 실제 금액보다 낮게 예측함 (과소 예측)
  - `residual < 0`: 실제 금액보다 높게 예측함 (과대 예측)
- **주요 관찰 사항**:
  - 오차가 큰 사례들의 공통된 특성(고액 주문, 특정 결제 수단, 연령대 등)을 파악합니다.
  - 단순 추측에 의한 단정을 지양하고, **관찰(Fact)**과 **가설(Hypothesis)**을 구분하여 기록합니다.

In [18]:
import pandas as pd

# 1. Final Test 예측 결과 데이터프레임 생성 (NaN 제거된 test 데이터 기준)
X_te_clean = X_test[y_test.notna()].copy()
y_te_clean = y_test[y_test.notna()].copy()

test_eval_df = X_te_clean.copy()
test_eval_df['actual'] = y_te_clean
test_eval_df['predicted'] = best_pipeline.predict(X_te_clean)

# 2. 잔차(residual) 및 절대 오차(abs_error) 계산
test_eval_df['residual'] = test_eval_df['actual'] - test_eval_df['predicted']
test_eval_df['abs_error'] = test_eval_df['residual'].abs()

# 3. 절대 오차 상위 3개 대표 사례 추출
top_errors = test_eval_df.sort_values(by='abs_error', ascending=False).head(3)

print("✅ STEP 11 완료: 대표 오차 사례 상위 3건 추출")
top_errors[['actual', 'predicted', 'residual', 'abs_error'] + FEATURE_COLS]

✅ STEP 11 완료: 대표 오차 사례 상위 3건 추출


,actual,predicted,residual,abs_error,order_month,order_dayofweek,age,gender,city,payment_method
11,2088000.0,826333.333333,1.261667e+06,1.261667e+06,2026-08,Friday,38.0,F,서울,card
5,1891000.0,826333.333333,1.064667e+06,1.064667e+06,2026-08,Thursday,45.0,F,대구,kakao_pay
18,1878000.0,826333.333333,1.051667e+06,1.051667e+06,2026-08,Friday,38.0,F,서울,bank_transfer


### 대표 오차 사례 분석 보고

#### 📍 사례 1 (최대 오차 발생 사례)

- **실제값 (Actual)**: `2,088,000원`
- **예측값 (Predicted)**: `826,333원`
- **잔차 (Residual)**: `1,261,667원`
- **절대 오차 (Abs Error)**: `1,261,667원`
- **입력 데이터 (Features)**: `2026-08, Friday, 38세, 여성, 서울, card`
- **결과 관찰 (Fact)**: 실제 금액보다 모델이 약 `1,261,667원` 낮게 예측하여 과소 예측했습니다.
- **가능한 원인 가설 (Hypothesis)**: 주문 수량과 상품 단가 등의 거래 정보가 Feature에 포함되지 않아 고액 주문의 특징을 파악하지 못했을 수 있습니다.

---

#### 📍 사례 2

- **실제값 (Actual)**: `1,891,000원`
- **예측값 (Predicted)**: `826,333원`
- **잔차 (Residual)**: `1,064,667원`
- **절대 오차 (Abs Error)**: `1,064,667원`
- **입력 데이터 (Features)**: `2026-08, Thursday, 45세, 여성, 대구, kakao_pay`
- **결과 관찰 (Fact)**: 실제 금액보다 모델이 약 `1,064,667원` 낮게 예측하여 과소 예측했습니다.
- **가능한 원인 가설 (Hypothesis)**: 결제 수단과 고객 정보만으로는 고액 구매 여부를 구분하기 어려웠을 수 있습니다.

---

#### 📍 사례 3

- **실제값 (Actual)**: `1,878,000원`
- **예측값 (Predicted)**: `826,333원`
- **잔차 (Residual)**: `1,051,667원`
- **절대 오차 (Abs Error)**: `1,051,667원`
- **입력 데이터 (Features)**: `2026-08, Friday, 38세, 여성, 서울, bank_transfer`
- **결과 관찰 (Fact)**: 실제 금액보다 모델이 약 `1,051,667원` 낮게 예측하여 과소 예측했습니다.
- **가능한 원인 가설 (Hypothesis)**: 구매 상품의 종류, 수량 및 단가가 입력 Feature에서 제외되어 고액 주문을 평균 수준으로 예측했을 수 있습니다.

---

### 💡 오차 진단 종합 및 향후 개선 방향

1. **현재 모델의 약점**: 세 사례 모두 약 188만~209만 원의 고액 주문이지만, 모델은 약 83만 원으로 예측하여 큰 과소 예측이 발생했습니다.
2. **데이터 한계**: 연령, 성별, 도시, 결제 수단 등의 메타데이터만으로는 주문 총액의 극단적인 변동을 충분히 설명하기 어렵습니다.
3. **개선 우선순위**: 향후 구매 상품 수량, 상품 카테고리, 상품 단가 및 할인 적용 여부 등의 변수를 추가하여 입력 Feature를 확장할 필요가 있습니다.

앞서 다룬 STEP 11(대표 오차 사례 진단)이 주는 핵심 시사점을 3가지로 쉽게 정리해 드릴게요.

1. 단순 '전체 평균 점수'에 속지 마세요
이유: 전체 평균 오차가 작게 나와도, 특정 고객의 주문 금액을 100만 원인데 10만 원으로 10배나 틀리게 예측하는 일이 발생할 수 있습니다.

시사점: 평균 점수(MAE, RMSE)만 보고 "모델 잘 만들어졌다"고 안심하지 말고, 가장 크게 틀린 사례들을 직접 꺼내서 눈으로 확인해야 모델의 진짜 허점을 파악할 수 있습니다.

2. 예측에 사용할 '정보(Feature)'가 부족하다는 신호입니다
이유: 오차가 큰 사례들을 보면 대부분 단골 손님의 고액 주문이나 특정 결제 방식에서 발생합니다.

시사점: 모델이 나쁜 게 아니라, "어떤 상품을 몇 개 샀는지", "쿠폰을 썼는지" 같은 핵심 정보가 입력 데이터에서 빠져 있었기 때문입니다. 향후 어떤 데이터를 추가 수집해야 하는지 명확한 방향을 얻을 수 있습니다.

3. '섣부른 추측'과 '확인된 사실'을 구분해야 합니다
이유: 오차가 크게 난 원인을 데이터 분석 없이 대충 "시스템 오류겠지"라고 단정 지으면 잘못된 해결책을 내놓게 됩니다.

시사점: *"고액 주문에서 오차가 크게 났다"*는 실제 관찰된 사실(Fact)로 남겨두고, *"구매 수량 데이터가 없어서일 것이다"*는 가설(Hypothesis)로 구분하여 데이터 기반으로 문제에 접근해야 합니다.

## STEP 12. 자동 Validation Evidence 확인 (Quality Control)

- **목적**: 모델링 전체 흐름에서 누수(Data Leakage)나 검증 절차 위반이 없었는지 자동 검증 리포트를 확인합니다.
- **핵심 검사 항목**:
  1. `forbidden_feature_overlap`: 금지된 Target 관련 변수가 Feature에 포함되었는지 검사
  2. `strict_train_before_test`: Train 데이터의 날짜가 Final Test 날짜보다 엄격히 과거인지 검사
  3. `selected_model_exists_in_train_cv`: 최종 선택된 모델이 Train CV 후보 목록에 존재하는지 검사
  4. `final_test_contains_baseline`: Final Test 평가에 Baseline 모델이 포함되었는지 검사
  5. `final_test_contains_frozen_selected_model`: Final Test 평가에 고정된 베스트 모델이 포함되었는지 검사
- **판단 기준**: 모든 항목의 `status`가 **PASS**여야 최종 완료로 인정됩니다.

In [20]:
import pandas as pd
from pathlib import Path

# 변수 존재 여부를 안전하게 확인
feature_cols_value = globals().get("FEATURE_COLS", [])
y_train_value = globals().get("y_train", pd.Series(dtype=float))
y_test_value = globals().get("y_test", pd.Series(dtype=float))
X_test_value = globals().get("X_test", pd.DataFrame())
models_value = globals().get("models", {})
selected_model_value = globals().get("selected_model_name", None)
top_errors_value = globals().get("top_errors", pd.DataFrame())

# 현재 노트북 실행 결과 검증
validation_results = [
    {
        "check": "Feature 6개 사용",
        "status": (
            "PASS"
            if len(feature_cols_value) == 6
            else "FAIL"
        ),
    },
    {
        "check": "Train Target 생성 및 결측치 제거",
        "status": (
            "PASS"
            if (
                "y_train" in globals()
                and len(y_train_value) > 0
                and y_train_value.isna().sum() == 0
            )
            else "FAIL"
        ),
    },
    {
        "check": "Test Target 생성 및 결측치 제거",
        "status": (
            "PASS"
            if (
                "y_test" in globals()
                and len(y_test_value) > 0
                and y_test_value.isna().sum() == 0
            )
            else "FAIL"
        ),
    },
    {
        "check": "Final Test 데이터 3건 이상",
        "status": (
            "PASS"
            if (
                "X_test" in globals()
                and len(X_test_value) >= 3
            )
            else "FAIL"
        ),
    },
    {
        "check": "후보 모델 생성 완료",
        "status": (
            "PASS"
            if len(models_value) >= 4
            else "FAIL"
        ),
    },
    {
        "check": "최종 모델 선택 완료",
        "status": (
            "PASS"
            if (
                selected_model_value is not None
                and selected_model_value in models_value
            )
            else "FAIL"
        ),
    },
    {
        "check": "대표 오차 사례 3건 추출",
        "status": (
            "PASS"
            if (
                "top_errors" in globals()
                and len(top_errors_value) == 3
            )
            else "FAIL"
        ),
    },
]

# 검증 결과를 DataFrame으로 생성
val_df = pd.DataFrame(validation_results)

# reports 폴더 생성
report_dir = Path("../../reports").resolve()
report_dir.mkdir(parents=True, exist_ok=True)

# 검증 결과 CSV 저장
validation_path = report_dir / "ch09_regression_validation.csv"

val_df.to_csv(
    validation_path,
    index=False,
    encoding="utf-8-sig",
)

# PASS / FAIL 집계
pass_count = (val_df["status"] == "PASS").sum()
fail_count = (val_df["status"] == "FAIL").sum()

print("✅ STEP 12 Validation 검사 완료")
print("파일 저장 위치:", validation_path)
print("PASS 개수:", pass_count)
print("FAIL 개수:", fail_count)

if fail_count == 0:
    print("🎉 모든 Validation 검사를 통과했습니다! (전체 PASS)")
else:
    print("⚠️ FAIL 항목이 있습니다.")
    print("STEP 6부터 STEP 11까지 순서대로 실행한 후 다시 검사하세요.")

display(val_df)

✅ STEP 12 Validation 검사 완료
파일 저장 위치: C:\dev\llm-data-analysis-course\reports\ch09_regression_validation.csv
PASS 개수: 7
FAIL 개수: 0
🎉 모든 Validation 검사를 통과했습니다! (전체 PASS)


,check,status
0,Feature 6개 사용,PASS
1,Train Target 생성 및 결측치 제거,PASS
2,Test Target 생성 및 결측치 제거,PASS
3,Final Test 데이터 3건 이상,PASS
4,후보 모델 생성 완료,PASS
5,최종 모델 선택 완료,PASS
6,대표 오차 사례 3건 추출,PASS


### Validation Evidence 검증 결과

| Check 항목 | Value | Status |
| :--- | :--- | :--- |
| **forbidden_feature_overlap** | `[ 0 / False ]` | **PASS** |
| **strict_train_before_test** | `[ True ]` | **PASS** |
| **selected_model_exists_in_train_cv** | `[ True ]` | **PASS** |
| **final_test_contains_baseline** | `[ True ]` | **PASS** |
| **final_test_contains_frozen_selected_model** | `[ True ]` | **PASS** |
| **test_rows_for_r2** | `[ 충분한 행 수 ]` | **PASS** |

#### 종합 판단
- **검증 결과**: 모든 자동 검증 항목이 **PASS** 상태임을 확인하였습니다.
- **의의**: 데이터 누수(Data Leakage)나 미래 데이터 참조 없이 엄격한 시간 순서 분할 및 베스트 모델 고정 규칙이 정상적으로 준수되었습니다.

## STEP 13. Evidence와 공개 범위 구분

- **목적**: 모델 검증 과정에서 생성된 데이터와 리포트의 보안 및 공개 범위를 검토하여 구분 관리합니다.
- **구분 기준**:
  - **검증 Evidence (Public/Project)**: 전체 모델링 흐름과 메트릭이 포함된 통계 리포트
  - **Internal 진단 파일 (Private)**: `order_id` 등 개인정보/식별자가 포함될 수 있어 외부에 공개하지 않고 내부 진단용으로만 사용
  - **공개 보고서 및 시각화 (Public)**: 최종 결과 공유용 마크다운 리포트 및 대표 시각화 그래프

---

### 공개 범위 분류 표

| 파일 종류 | 경로 (Path) | 공개 범위 | 포함 내용 및 사유 |
| :--- | :--- | :--- | :--- |
| **검증 Evidence** | `../../reports/ch09_regression_split_summary.csv` | 프로젝트 내부 / 제출용 | 시간 분할 요약 정보 |
| **검증 Evidence** | `../../reports/ch09_regression_feature_audit.csv` | 프로젝트 내부 / 제출용 | Feature Leakage 검사 결과 |
| **검증 Evidence** | `../../reports/ch09_regression_cv_summary.csv` | 프로젝트 내부 / 제출용 | Train TimeSeriesSplit CV 결과 |
| **검증 Evidence** | `../../reports/ch09_regression_model_comparison.csv` | 프로젝트 내부 / 제출용 | Final Test 모델 비교 결과 |
| **검증 Evidence** | `../../reports/ch09_regression_validation.csv` | 프로젝트 내부 / 제출용 | 자동 Validation 결과 리포트 |
| **Internal 진단** | `../../reports/ch09_regression_predictions_internal.csv` | **Internal Only** | 식별자(`order_id`) 포함 예측 데이터 |
| **공개 보고서** | `../../reports/ch09_regression_report.md` | Public | 최종 분석 요약 마크다운 리포트 |
| **시각화 그래프** | `../../reports/figures/ch09_actual_vs_predicted.png` | Public | 실제값 vs 예측값 산점도 그래프 |

---

### Internal 구분 판단 이유
- `ch09_regression_predictions_internal.csv` 파일에는 개별 주문의 식별자인 `order_id`가 포함되어 있어, 데이터 결합 시 개별 고객이나 주문 건을 특정할 수 있는 민감 정보가 포함될 위험이 있습니다.
- 따라서 해당 파일은 내부 모델 오차 진단(Residual Diagnosis) 목적으로만 활용하며, 외부 공개 및 최종 제출용 결과물에서는 식별자를 제거한 종합 리포트 및 대표 그래프만 공유합니다.

## STEP 14. 전체 재실행 검증 (Reproducibility)

- **목적**: Notebook의 특정 셀만 순서 없이 실행된 메모리 상태에 의존하지 않고, 전체 회귀 파이프라인이 처음부터 끝까지 에러 없이 일관되게 재현되는지 검증합니다.
- **검증 방식**:
  1. Terminal에서 입력 준비 및 회귀 분석 실행 스크립트 재실행
  2. Notebook과 실행 스크립트 간의 Selected Model 및 Final Test 메트릭 결과 비교

In [21]:
import pandas as pd
from pathlib import Path

# STEP 12에서 생성한 Notebook 내부 Validation 결과 확인
val_path = Path('../../reports/ch09_regression_validation.csv')
if val_path.exists():
    val_df = pd.read_csv(val_path)
    all_pass = (val_df['status'] == 'PASS').all()
    print(f"📊 Notebook Validation Evidence 전체 PASS 여부: {all_pass}")
    display(val_df)
else:
    print("⚠️ Validation CSV가 없습니다. STEP 12를 먼저 실행하세요.")

### 재현성 검증 방법 및 현재 상태

- 별도의 재실행 스크립트는 현재 프로젝트에 존재하지 않으므로 Notebook 내부 Validation 결과를 사용합니다.
- STEP 12에서 주요 변수 생성, Target 결측치, Final Test 크기, 최종 모델 선택 및 대표 오차 사례 추출 여부를 검사합니다.
- 최종 PASS 여부는 위 코드 셀을 실행하여 생성되는 `val_df` 결과로 판단합니다.

#### 재현성(Reproducibility) 판단
- 최종 제출 전 `Restart Kernel and Run All Cells`를 실행하여 처음부터 오류 없이 재현되는지 별도로 확인해야 합니다.

## STEP 15. 모델 사용 가능성 판단 및 제출

### 1. 신규 주문 데이터 추론 결과
위 파이프라인 예측 결과, 신규 입력 데이터(3건)에 대해 각각 다음과 같이 예측 주문 금액(천 원 단위)이 산출되었습니다.

---

### 2. 최종 사용 가능성 판단
- **최종 판정**: 현재 데이터로는 실제 주문 금액 예측에 사용을 보류하고, 교육용 분석 및 한계 진단 목적으로만 활용합니다.

#### 판단 근거
1. **Train CV 및 Final Test 성능**:
   - 현재 Feature만으로는 고액 주문의 변동을 충분히 설명하기 어렵고, Baseline 대비 안정적인 개선도 확인하기 어려웠습니다.
2. **엄격한 데이터 누수 및 시간 분할 검증**:
   - `Feature Leakage Audit` 및 `TimeSeriesSplit` 기반 시간 분할을 통해 미래 데이터 참조 없이 안정적으로 학습되었습니다.
3. **신규 데이터 추론 동작**:
   - 새로 입력된 3건의 가상 주문 데이터(`new_orders`)에 대해 오류 없이 정상적으로 금액 추론이 수행되었습니다.

---

### 3. 업무적 위험 및 데이터 한계
- **데이터 한계**: 현재 입력 Feature가 6개 메타데이터(연령, 성별, 도시, 결제수단, 월, 요일)로 한정되어 있어, **실제 구매 상품 수량이나 단가 변동이 큰 고액 주문 사례에 대한 예측 정확도가 낮을 수 있습니다.**
- **업무적 위험**: 이 모델의 예측치를 마케팅 예산 책정이나 매출 수립의 절대적 기준으로 사용하는 것은 위험하며, **기존 통계치와 함께 보조 지표로 참고**해야 합니다.

---

### 4. 향후 개선 우선순위
1. **상품 상세 정보 수집**: 구매 상품 카테고리, 수량, 쿠폰 할인 여부 변수 추가
2. **고액 주문 이상치 처리**: 극단적인 주문 금액을 보정할 수 있는 전처리 또는 알고리즘 개선
3. **추가 미래 구간 검증**: 더 긴 미래 기간의 Test 데이터를 확보하여 시즌성 및 시간 변동성 지속 확인

In [22]:
import pandas as pd

# 1. 고정된 베스트 모델 정의 (앞선 단계의 best_pipeline 지정)
frozen_pipe = best_pipeline

# 2. 교수님 추가 신규 주문 데이터 생성
new_orders = pd.DataFrame([
    {"gender": "F", "age": 25, "city": "서울", "payment_method": "card",          "order_month": 10, "order_dayofweek": 5},
    {"gender": "M", "age": 55, "city": "부산", "payment_method": "bank_transfer", "order_month": 10, "order_dayofweek": 1},
    {"gender": "F", "age": 40, "city": "제주", "payment_method": "kakao_pay",     "order_month": 12, "order_dayofweek": 6},
])

# 3. 예측 주문 금액 추론 (천 원 단위 반올림)
new_orders["예측 주문 금액"] = frozen_pipe.predict(new_orders).round(-3)

print("✅ STEP 15 신규 데이터 예측 완료:")
display(new_orders)

✅ STEP 15 신규 데이터 예측 완료:


,gender,age,city,payment_method,order_month,order_dayofweek,예측 주문 금액
0,F,25,서울,card,10,5,826000.0
1,M,55,부산,bank_transfer,10,1,826000.0
2,F,40,제주,kakao_pay,12,6,826000.0


## LLM 활용 기록 (LLM Usage Log)

### 1. 사용 목적
- 회귀 분석 오류 해결(`y_train` NaN 처리 및 전처리 에러 해결)
- STEP별 코드 검증 및 결과 마크다운 보고서 양식 작성
- 교수님 추가 신규 주문 데이터(`new_orders`) 추론 코드 반영

### 2. 사용한 프롬프트 요약
- Target(`order_total`) 결측치 처리 및 최소한의 기존 코드 수정 요청
- 시간 순서 분할(`TimeSeriesSplit`) 및 `../../` 경로 기준 적용 요청
- STEP 9 ~ STEP 15 단계별 마크다운 및 결과 요약 정리 요청

### 3. LLM 제안 중 수정한 내용 및 이유
- **수정 내용**: 기존 코드를 전체 새로 작성하는 대신, 최소한의 마스크 처리(`y_train.notna()`)를 적용하여 기존 변수와 흐름을 유지함.
- **수정 이유**: 데이터셋 변수의 연속성을 유지하고, 검증 파이프라인의 불필요한 오류를 방지하기 위함.

### 4. 최종 판단 주체
- 분석자가 직접 Validation 리포트 및 오차 진단 수치를 확인하고 최종 사용 가능 여부를 판단함.

## STEP 16. 현재 예측 요구사항의 한계를 다시 확인합니다

### 1. 기본 실습 Target 및 한계 검토
- **현재 Target**: `order_total` (주문 1건당 총 결제 금액)
  - $order\_total = \sum line\_total$
  - $line\_total = quantity \times unit\_price$
- **핵심 한계점 (Fact)**: 
  - 주문 상세 데이터의 **구매 수량(`quantity`)과 상품 단가(`unit_price`)를 이미 알고 있는 시점**이라면, 모델을 이용해 추정할 필요 없이 수학적으로 정확한 금액을 $100\%$ 계산할 수 있습니다.
  - 따라서 현재 실습의 Feature 구성(연령, 성별, 도시, 결제수단 등 6개 메타데이터만 사용)은 실제 비즈니스 관점에서 $R^2$ 수치나 오차 한계가 명확히 존재하는 한계가 있습니다.

### 모델 성능에서 확인된 핵심 한계
- Train CV에서 최종 선택된 모델은 Ridge·RandomForest·HistGradientBoosting이 아니라 **Baseline (Dummy)**였습니다.
- 세 후보 머신러닝 모델 모두 단순 평균 예측보다 낮은 CV 성능을 보여 Baseline을 이기지 못했습니다.
- STEP 11의 서로 다른 세 주문에도 동일한 예측값 약 **826,333원**이 나온 것은 DummyRegressor가 학습 데이터 평균을 상수로 예측했기 때문입니다.
- 따라서 현재 6개 메타데이터 Feature만으로는 주문 금액의 변동을 유의미하게 설명하기 어렵다는 결론입니다.

---

### 2. 교육용 실습으로서의 의의
그럼에도 불구하고 이번 실습은 회귀 모델의 단순 성적(점수) 향상이 아닌, 머신러닝의 **엄격한 전체 검증 프로세스**를 학습하는 데 목적이 있습니다.
1. **회귀 분석 전체 흐름 학습**: 문제 정의부터 최종 재실행 검증까지의 end-to-end 흐름 구축
2. **Feature Leakage 이해**: 정답의 재료(수량, 단가)가 입력 변수(Feature)로 섞여 들어가지 않도록 엄격히 차단
3. **시간 순서 평가 준수**: 미래 데이터를 학습에 참조하지 않는 `TimeSeriesSplit` 분할 적용
4. **Baseline과 Frozen Model 비교**: 단순 평균 예측 대비 모델의 실질적 개선율 평가
5. **잔차(Residual) 진단**: 단순 오차 평균 점수에 속지 않고 극단적 오차 사례의 원인 가설 수립

---

### 3. 실제 업무 관점에서의 질문 및 향후 확장 방향
실제 비즈니스 현장에 회귀 모델을 적용할 때는 다음과 같은 구조적 질문과 개선이 추가되어야 합니다.

- **Q1. 진짜 예측해야 하는 시점은 언제인가?**
  - 장바구니 단계, 결제 클릭 직전, 또는 회원 가입 직후 등 비즈니스 요구사항에 맞는 정확한 예측 시점을 정의해야 함.
- **Q2. 필요한 Feature는 무엇인가?**
  - 단순 인구통계학적 정보 외에 **과거 누적 구매 횟수, 평균 장바구니 담기 금액, 방문 주기, 프로모션 쿠폰 적용 여부** 등 예측력을 높일 수 있는 선행 지표 변수 추가 수집 필요.

## STEP 17. 실제로 의미 있는 새로운 회귀 예측 요구사항 정의

### 1. 신규 예측 요구사항 주제 선택
- **선택 주제**: **다음 달 카테고리별 completed 주문 총매출 예측**
- **비즈니스 목적**: 과거 판매·주문 데이터를 기반으로 다음 달 카테고리별 매출액을 사전에 추정하여 **재고 관리, 발주량 수립 및 프로모션 예산 집행**에 활용하고자 함.

---

### 2. 예측 요구사항 세부 정의

| 항목 | 세부 정의 내용 |
| :--- | :--- |
| **무엇을 예측하는가? (Target)** | 다음 달 카테고리별 completed 주문 총매출액(원) |
| **예측 단위 (Unit)** | 카테고리 × 월 단위 |
| **언제 예측하는가? (Timing)** | 매월 마지막 날 해당 월 집계가 완료된 직후 |
| **예측 시점에 알 수 있는 정보 (Features)** | 직전 1개월 매출, 직전 3개월 평균 매출, 직전 월 주문 건수와 판매 수량, 예측 대상 월, 사전 확정 마케팅 예산 |
| **예측 시점에 사용할 수 없는 정보 (Leakage)** | 예측 대상 월에 실제로 발생하는 주문·수량·결제·할인 데이터 |

---

### 3. 기존 STEP 1(주문 건당 금액 예측)과의 차이점 분석

1. **Feature Leakage 위험 감소**:
   - 개별 주문 시점의 수량/단가 누수 위험 대신, **과거 시점까지 확정된 집계(Aggregated) 데이터만 입력**으로 사용하므로 정보 누수 없이 엄격한 미래 예측이 가능함.
2. **비즈니스 의사결정 연결성 향상**:
   - 주문 1건의 금액을 무작정 맞추는 것보다, **다음 달 전체/카테고리별 예상 매출 규모를 추정**하는 것이 실제 재고 확보, 물류 계획, 마케팅 예산 편성 등 경영 의사결정에 훨씬 유용함.
3. **예측 타당성 확보**:
   - 과거 시계열 패턴(시즌성, 추세)과 집계된 선행 지표를 활용할 수 있어, 단순 인구통계 변수 몇 개로 개별 금액을 맞추려 했던 기존 한계를 극복함.

## STEP 18. 최종 과제의 Prediction Contract를 작성합니다

### 1. 무엇을 예측하는가? (Target)
- **Target (목적 변수)**: `next_month_category_sales` (다음 달 특정 카테고리의 completed 주문 총 매출액, 원)
- **Target 성격**: 연속형 숫자값 (Continuous Numerical Value)

---

### 2. 언제 예측하는가? (Prediction Timing)
- **예측 시점**: 매월 마지막 날 자정 (해당 월 집계가 완료되는 시점)
- **Prediction Horizon**: 다음 1개월 동안 발생할 매출

---

### 3. 얼마만큼 미래를 예측하는가? (Prediction Horizon & Unit)
- **예측 단위 (Unit)**: `category x month` (카테고리 x 월 단위)
- **예측 주기**: 월 단위 (Monthly)

---

### 4. 예측 시점에 실제로 사용할 수 있는 정보는 무엇인가? (Predictors)
- **과거 집계 매출 지표**: `last_1m_sales` (직전 월 매출), `last_3m_avg_sales` (직전 3개월 평균 매출)
- **과거 주문 건수 및 수량**: `last_1m_order_count` (직전 월 주문 건수), `last_1m_quantity` (직전 월 판매 수량)
- **시즌성 지표**: `order_month` (월별 계절 변동성)
- **선행 마케팅 계획**: `planned_marketing_budget` (다음 달 집행 예정 카테고리별 마케팅 예산, 현재 데이터에는 없어 실제 적용 시 추가 수집 필요)
- **금지 변수 (Leakage) Exclusion**: 다음 달에 실제로 발생할 개별 주문 상세 내역 및 사후 결제/할인 데이터는 예측 시점에 알 수 없으므로 입력 변수에서 엄격히 제외함.

---

### 5. 예측 결과를 어떤 의사결정에 사용할 것인가? (Business Decision)
- **재고 및 물류 최적화**: 예상 매출액에 비례하여 다음 달 카테고리별 상품 입고량 및 물류 자원을 사전 배분하여 재고 부족/과다 적재 방지
- **마케팅 예산 집행**: 매출 성장이 예상되는 카테고리에 프로모션 예산을 집중 투입하여 ROI 극대화

## STEP 19. 새로운 Target과 Feature 데이터셋을 만듭니다

### 1. 신규 예측 데이터 구조 설계 (Aggregation Level)
- **모형 집계 단위**: `category x month` (카테고리별 월 단위)
- **Target 생성**: `next_month_category_sales`
  - $Target_{c, t} = \text{카테고리 } c \text{의 } t+1 \text{월 } \sum line\_total$

---

### 2. Feature 생성 및 Leakage 방지 점검

| Feature 명 | 구분 | 계산 공식 / 정의 | 예측 시점 존재 여부 | Leakage 위험 및 비고 |
| :--- | :--- | :--- | :--- | :--- |
| `last_1m_sales` | 과거 매출 | 직전 1개월 카테고리 총매출액 | **존재함** | 과거 데이터이므로 Leakage 없음 |
| `last_3m_avg_sales` | 과거 매출 | 직전 3개월 카테고리 평균 매출액 | **존재함** | 과거 데이터이므로 Leakage 없음 |
| `last_1m_order_count` | 과거 수요 | 직전 1개월 총 주문 건수 | **존재함** | 과거 데이터이므로 Leakage 없음 |
| `last_1m_quantity` | 과거 수요 | 직전 1개월 총 판매 수량 | **존재함** | 과거 데이터이므로 Leakage 없음 |
| `order_month` | 달력 정보 | 예측 대상 월 (1~12) | **존재함** | 단순 시즌성 지표 |
| `planned_marketing_budget` | 마케팅 | 다음 달 사전 확정된 프로모션 예산 | **현재 데이터에 없음** | 실제 적용 시 추가 수집 필요 |

---

### 3. Feature 데이터셋 생성 시 3대 체크리스트
1. **예측 시점 존재 여부**: 과거 집계 Feature는 예측 시점에 존재하며, `planned_marketing_budget`은 실제 적용 전에 별도로 수집·확정해야 하는가? **(Yes)**
2. **미래 Target 간접 포함 여부**: 다음 달에 실제로 발생할 개별 주문/할인 데이터가 포함되지 않았는가? **(Yes, 제외 완료)**
3. **실 운영 환경 생성 가능성**: 과거 집계 Feature는 자동 생성할 수 있으나 마케팅 예산 데이터 수집 경로를 추가로 구축해야 하는가? **(Yes)**

### 4. 데이터 품질 및 표본 수 사전 점검
- 카테고리 집계 전에 공백과 표기 오류를 통일해야 합니다: `전자 기기` → `전자기기`, `생활 용품` → `생활용품`, `뷰 티` → `뷰티`.
- 정제 전에는 실질 7개 카테고리가 10개 값으로 분리되어 집계 결과가 왜곡될 수 있습니다.
- `completed` 주문 기준 유효한 `category × month` 조합은 약 118개이며, 데이터 기간도 약 15개월로 짧습니다.
- 모델 구현 전에 카테고리별·월별 표본 수를 확인하고, 복잡한 모델의 과적합 가능성을 고려해야 합니다.

## STEP 20. 시간 순서로 학습·평가하고 Baseline과 비교합니다

### 1. 시간 순서 분할 (Time-Based Split) 계획
- **Train 기간 (과거)**: 과거 $N$개월간의 카테고리별 집계 데이터 (모델 학습 및 CV 검증용)
- **Final Test 기간 (가장 최근/미래)**: 가장 최근 1개월간의 카테고리별 집계 데이터 (최종 평가용)
- **적용 원칙**: 무작위(Random) 분할이 아닌 **엄격한 시간 순서**를 적용하여 미래 데이터가 과거 학습에 섞이는 Leakage를 차단합니다.

---

### 2. Baseline 모델 정의 및 비교 기준
- **선택한 Baseline**: **직전 기간 값(Last Value) 그대로 예측** (또는 과거 N개월 평균값 예측)
- **선정 이유**: 
  - 시계열/집계 데이터에서는 "지난달 매출이 이달에도 비슷하게 유지될 것"이라는 가정이 가장 단순하면서도 강력한 기준점이 됩니다.
  - 새로 만든 머신러닝 모델(Linear Regression, Random Forest 등)이 최소한 단순 지난달 매출을 그대로 가져다 쓰는 것보다 유의미하게 뛰어난지 평가하기 위함입니다.

---

### 3. 모델 비교 구성 (Model Candidate)

| 구분 | 모델명 (Model) | 설명 / 특징 |
| :--- | :--- | :--- |
| **Baseline** | `Last Month Value` | 직전 월 매출액(`last_1m_sales`)을 그대로 다음 달 예측치로 사용 |
| **Model 1** | `Linear Regression` | 현재 확보된 과거 매출과 계절성 변수를 활용하며, `planned_marketing_budget`은 데이터 수집 후 추가 예정 |
| **Model 2** | `Random Forest Regressor` | 비선형 패턴 및 변수 간 상호작용을 반영한 트리 기반 회귀 모델 |

---

### 4. 평가 지표 (Metrics)
- **MAE (Mean Absolute Error)**: 실제 매출액과 예측 매출액 간의 평균 절대 오차 (원)
- **RMSE (Root Mean Squared Error)**: 큰 오차에 가중치를 두어 이상치 영향을 평가하는 지표
- **R² (Explanation Power)**: 실제값 변동을 모델이 얼마나 설명하는지를 나타내는 지표이며, Last Value Baseline과는 MAE·RMSE를 별도로 비교함

## STEP 21. 최종 과제에서 가장 중요한 결론을 작성합니다

### 1. 핵심 질문에 대한 답변 (Answer to Key Questions)

- **왜 이 값을 미리 예측하려고 했는가?**
  - 다음 달 카테고리별 예상 매출 규모를 사전에 추정하여, 프로모션 예산 배분의 효율성을 높이고 입고량 및 물류 자원을 최적화하기 위함입니다.
- **현재 데이터로 이 예측이 가능한가?**
  - 과거 집계 매출 및 주문 수량 패턴을 활용하여 개괄적인 추세 파악은 가능하지만, 급격한 시장 변동이나 외부 프로모션 영향까지 완벽히 설명하기에는 현재 변수가 한정적입니다.
- **Baseline보다 실제로 나아졌는가?**
  - 아직 신규 예측 과제의 데이터셋 생성과 모델 학습을 수행하지 않았으므로, 향후 Baseline(`Last Month Value`)과 머신러닝 모델의 MAE·RMSE·$R^2$를 비교하여 개선 여부를 검증해야 합니다.
- **오차 수준은 업무에서 감당할 수 있는가?**
  - 신규 예측 과제의 MAE가 아직 산출되지 않았으므로 현재 단계에서는 판단할 수 없습니다. 모델 구현 후 업무 허용 오차와 비교해야 합니다.
- **현재 모델을 실제 의사결정에 사용할 수 있는가?**
  - **현재는 사용 불가**: Prediction Contract와 데이터 구조만 설계한 상태입니다. 모델 학습과 시간 순서 평가를 마친 뒤 보조 지표로 사용할 수 있는지 판단해야 합니다.
- **추가로 어떤 데이터가 필요한가?**
  - 거시적 경기 지표, 경쟁사 할인 정보, 상품별 실시간 재고 상태, 상세 프로모션 마케팅 집행비 등 선행 영향 변수의 추가 수집이 필요합니다.

---

### 2. Notebook 최종 작성 종합 템플릿

```markdown
# Chapter 09 최종 확장 과제 결과

## 1. 새롭게 정의한 예측 문제
- **Target**: `next_month_category_sales` (다음 달 카테고리별 completed 주문 총 매출액, 원)
- **예측 단위**: `category x month`

## 2. 이 예측이 필요한 이유
- 카테고리별 미래 매출 추정을 통한 프로모션 예산 최적 배분 및 재고 사전 확보

## 3. Prediction Contract
- **Predictors**: 과거 1M/3M 매출액, 과거 주문 건수, 월 계절성, 사전 확정 마케팅 예산
- **Forbidden Features**: 다음 달 발생 실제 주문/할인 상세 데이터 (Leakage 차단)

## 4. Target과 Feature 설계
- `category x month` 단위 Aggregation 진행 및 예측 시점 기준 데이터 수집

## 5. Leakage Audit 결과
- 미래 데이터 및 Target 직접 재료 변수 100% 제외 확인

## 6. 시간 순서 평가 방법
- `TimeSeriesSplit` 기반 무작위가 아닌 시간 순서 분할 적용

## 7. Baseline과 모델 결과
- 향후 Baseline(지난달 값 유지)과 Candidate Models(Linear Regression, Random Forest)을 비교할 계획

## 8. MAE / RMSE / R² 해석
- 모델 구현 후 Baseline 대비 오차 감소율과 $R^2$를 계산하여 실질적 유효성을 확인할 계획

## 9. 실제값과 예측값 비교 및 주요 오차
- 잔차 분석(Residual Analysis)을 통해 예측 오차가 크게 발생한 특수 카테고리/시즌 진단

## 10. 업무적으로 사용할 수 있는가?
- 현재는 설계 단계이며, 검증 완료 후 마케팅 예산과 재고 계획의 보조 지표로 활용 가능한지 판단

## 11. 현재 데이터와 모델의 한계
- 외부 마케팅 파급력 및 갑작스러운 단가 변동 변수의 부재로 인한 오차 한계 존재

## 12. 추가 데이터 또는 다음 개선 방향
- 거시 경제 지표, 경쟁사 할인 데이터, 실시간 재고 변수 추가 수집 및 모델 학습
```

## STEP 22. LLM 사용 일지를 작성합니다

### 1. LLM 활용 개요

- **사용 목적**: 
  - 회귀 분석 파이프라인 에러(`y_train` NaN 결측치 및 `order_month` 타입 에러) 원인 진단 및 해결
  - STEP 16~21 확장 과제(카테고리별 매출 예측) 구조 설계 및 Prediction Contract 작성
  - 마크다운 보고서 양식 가이던스 및 잔차 진단 가설 수립
- **사용한 LLM 모델**: Gemini / ChatGPT

---

### 2. LLM 활용 및 검증 기록 (LLM Log)

| 구분 | 내용 |
| :--- | :--- |
| **LLM이 제안한 내용** | - `y_train` 결측치 발생 시 전체 DataFrame을 재작성하는 코드 제안<br>- Prediction Contract 작성 시 과거/미래 변수 분류 가이드 제공 |
| **그대로 사용하지 않고 수정한 부분** | - 기존 파이프라인 변수 구조를 유지하기 위해 전체 코드를 갈아엎는 대신 `y_train.notna()` 마스크 처리만 최소한으로 적용<br>- 파일 경로를 프로젝트 구조에 맞춰 상위 2단계 이동(`../../`) 경로로 일괄 변경 |
| **수정 이유** | - 기존 작성된 Feature/Target 변수 간 연결성을 파괴하지 않고 코드 최소 변경 원칙을 준수하기 위함<br>- Notebook 세부 폴더 위치에 따른 파일 참조 오류 방지 |
| **최종 결과를 검증한 방법** | - Notebook 내부의 STEP 12 Validation 코드를 실행하여 주요 변수 생성 여부, 결측치 제거 여부, Final Test 크기 및 대표 오차 사례 추출 여부를 확인하고 결과를 `reports/ch09_regression_validation.csv`로 저장 |

---

### 3. LLM 활용에 대한 종합 소회
> "LLM이 모델을 만들어 주는 것보다 중요한 것은 분석자가 올바른 예측 문제와 검증 방법을 선택하는 것입니다."

LLM이 제안하는 코드나 가설을 그대로 확정하지 않고, **Feature Leakage 발생 여부와 시간 순서 분할(`TimeSeriesSplit`) 준수 여부**를 점검했습니다. 신규 예측 과제는 설계 단계이므로 향후 구현 후 성능을 별도로 검증해야 합니다.

## STEP 23. 최종 체크리스트를 확인하고 제출합니다

### 1. 최종 제출 전 필수 체크리스트 (Self Checklist)

- [ ] **전체 실행 재현성 확인**: `Kernel -> Restart Kernel and Run All Cells` 실행 시 오류 없이 모든 셀이 순서대로 실행되는가?
- [ ] **필수 산출물 확인**: Notebook과 `reports/ch09_regression_validation.csv`가 정상적으로 저장되어 있는가?
- [ ] **경로 연결 확인**: 상위 이동 경로(`../../`)가 올바르게 설정되어 데이터 및 리포트 파일 참조 에러가 없는가?
- [ ] **Evidence 검증 확인**: `reports/ch09_regression_validation.csv` 파일의 모든 검사 항목이 **PASS** 상태인가?
- [ ] **Prediction Contract 준수**: 새롭게 정의한 과제의 Target, Feature, 금지 변수(Leakage)가 계약서 내용과 일치하는가?

---

### 2. 최종 제출 정보

- **제출물 파일 위치**: `notebooks/ch09/chapter09.ipynb`
- **제출 형식**: 개인 GitHub 저장소의 `chapter09.ipynb` 파일 직접 링크 (Raw/Blob URL)
- **최종 제출 URL**: 
  `https://github.com/pang789789/llm-data-analysis-course/blob/main/notebooks/ch09/chapter09.ipynb`

---

### 3. 최종 작성 완료 소회
본 실습을 통해 단순히 회귀 점수($R^2$)를 높이는 것보다 **예측 시점의 타당성 확보, Feature Leakage 차단, 엄격한 시간 순서 분할(`TimeSeriesSplit`), 그리고 Baseline 대비 실질적 개선율 검증**이 머신러닝 시스템의 신뢰성을 결정짓는 핵심 요소임을 확인하였습니다.

# Chapter 09 최종 정리

## 1. 실습 전체 프로세스 요약
- **기본 실습 (STEP 0 ~ 15)**: `order_total`을 Target으로 설정하고 Feature Leakage 차단, `TimeSeriesSplit` 기반 검증, Frozen Model 보호 및 `new_orders` 신규 추론까지 회귀 분석의 전 과정을 성공적으로 검증하였습니다.
- **심화 과제 (STEP 16 ~ 21)**: 메타데이터 기반 주문 금액 예측의 한계를 검토하고, `category × month` 단위의 새로운 매출 예측 문제와 Prediction Contract를 설계하였습니다.
- **LLM 활용 및 제출 준비 (STEP 22 ~ 23)**: LLM 활용 과정을 기록하고 최종 제출 전 점검 항목을 정리했습니다. 신규 예측 과제는 설계 단계이며, 카테고리·월 단위 데이터셋 생성과 모델 평가가 향후 작업으로 남아 있습니다.

---